# SeqComm — Island Coverage Ablation

**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Clone repo and install deps
import os
REPO = 'multi-lvl-comms'
if not os.path.exists(REPO):
    !git clone https://github.com/bcupps9/multi-lvl-comms.git
else:
    !git -C {REPO} pull
os.chdir(REPO)
!pip install wandb --quiet
import torch
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

In [ ]:
# 2. W&B login — get key from https://wandb.ai/authorize
# Skip this cell if you don't want W&B tracking.
import wandb
wandb.login()

In [ ]:
# 3. Config
import os
MODES    = ['seqcomm', 'mappo', 'seqcomm_random', 'seqcomm_no_action', 'seqcomm_fixed']
SEEDS    = [0, 1]
EPISODES = 750
LOG_DIR  = 'logs/island_colab'
USE_WANDB = True
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs('figures', exist_ok=True)
print(f'{len(MODES)} modes x {len(SEEDS)} seeds x {EPISODES} episodes')

In [ ]:
# 4. Launch runs with a capped pool (avoids OOM from simultaneous PyTorch processes)
# Jobs interleaved seed-first so all modes get slots early.
import subprocess, sys, time

MAX_CONCURRENT = 4

jobs = []
for seed in SEEDS:
    for mode in MODES:
        jobs.append((mode, seed))

active = []
done_count = 0
total = len(jobs)
start = time.time()

def launch_next():
    if not jobs:
        return
    mode, seed = jobs.pop(0)
    log_file = open(f'{LOG_DIR}/{mode}_seed{seed}.stdout', 'w')
    cmd = [
        sys.executable, '-m', 'training.train',
        '--env', 'coverage',
        '--mode', mode,
        '--seed', str(seed),
        '--episodes', str(EPISODES),
        '--log-dir', LOG_DIR,
        '--wm-H', '1', '--wm-F', '1',
        '--episode-len', '50',
        '--wandb-project', 'multi-lvl-comms',
    ] + (['--wandb'] if USE_WANDB else [])
    p = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
    active.append((mode, seed, p, log_file))
    print(f'  -> launched {mode} seed={seed} (pid {p.pid})  [{len(active)}/{MAX_CONCURRENT} slots]')

while jobs and len(active) < MAX_CONCURRENT:
    launch_next()

while active:
    time.sleep(30)
    still_running = []
    for item in active:
        mode, seed, p, f = item
        if p.poll() is not None:
            f.close()
            done_count += 1
            print(f'  done: {mode} seed={seed}  [{done_count}/{total} total]')
            launch_next()
        else:
            still_running.append(item)
    active[:] = still_running

    mins = (time.time() - start) / 60
    ep_counts = []
    for m in MODES:
        try:
            n = sum(1 for l in open(f'{LOG_DIR}/coverage_{m}_seed0.jsonl') if '"episode"' in l)
        except FileNotFoundError:
            n = 0
        ep_counts.append(f'{m[:10]}:{n}')
    print(f'[{mins:.1f}m] {done_count}/{total} done | ' + '  '.join(ep_counts))

print('All done!')

In [ ]:
# 5. Plot results inline
import subprocess, sys
r = subprocess.run(
    [sys.executable, 'scripts/plot_island.py', '--log-dir', LOG_DIR, '--out', 'figures/island_colab.png'],
    capture_output=True, text=True
)
print(r.stdout or r.stderr)
from IPython.display import Image, display
display(Image('figures/island_colab.png'))

In [ ]:
# 6. Summary table: final 50-episode average per mode
import json, glob
print(f'{"Mode":<25} {"Coverage rate":>15} {"Reward":>12}')
print('-' * 54)
for mode in MODES:
    files = sorted(glob.glob(f'{LOG_DIR}/coverage_{mode}*.jsonl'))
    covs, rews = [], []
    for fpath in files:
        recs = [json.loads(l) for l in open(fpath) if '"episode"' in l]
        if recs:
            tail = recs[-50:]
            covs.append(sum(r['coverage_rate'] for r in tail) / len(tail))
            rews.append(sum(r['total_reward']  for r in tail) / len(tail))
    if covs:
        print(f'{mode:<25} {sum(covs)/len(covs):>15.3f} {sum(rews)/len(rews):>12.1f}')